# Python Development Environments

This module covers the tools and workflows professional Python developers use every day to manage packages, isolate project environments, work with version control, and keep builds reproducible.

## Modules

| # | Topic | What you'll learn |
|---|---|---|
| 01 | [Managing Python Packages with pip](./01-managing-python-packages-with-pip/managing-python-packages-with-pip.ipynb) | Install, upgrade, inspect, and freeze packages |
| 02 | [Virtual Environments and Project Dependencies](./02-virtual-environments-and-project-dependencies/virtual-environments-and-project-dependencies.ipynb) | Isolate projects with `venv`; apps vs. library dependency strategies |
| 03 | [Python Environments and Version Control](./03-python-environments-and-version-control/python-environments-and-version-control.ipynb) | What to commit and what to ignore; secrets management with `.env` |
| 04 | [Best Practices and Newer Tools](./04-best-practices-and-newer-tools/best-practices-and-newer-tools.ipynb) | Lockfiles, `uv`, `poetry`, and when to use each |

## Prerequisites

- Basic Python syntax (variables, functions, imports)
- Comfort running commands in a terminal
- No prior knowledge of package managers or virtual environments needed

## Why this matters

Every Python project eventually runs into dependency problems:

- *"It works on my machine"* — but not in CI or on a teammate's laptop
- A package upgrade breaks an unrelated part of the app
- API keys end up committed to git by accident

The tools and practices in this module exist specifically to prevent those problems. By the end you'll be able to set up any Python project so it's isolated, reproducible, and safe to share.

## Choosing an environment manager

Python has several tools that overlap in purpose. Use this guide to pick the right one without having to read everything first.

### The landscape at a glance

| Tool | What it manages | Lockfile | Best for |
|---|---|---|---|
| `pip` + `venv` | packages, one env | `requirements.txt` (manual) | Learning, simple scripts |
| `pip` + `venv` + `pip-tools` | packages, one env | `requirements.txt` (generated) | Teams that want reproducibility without switching tools |
| `poetry` | packages + env + publishing | `poetry.lock` | Libraries you publish to PyPI |
| `uv` | packages + env + Python version | `uv.lock` | Any new project; fastest option available |
| `conda` / `mamba` | packages + env + non-Python deps | `environment.yml` | Data science, ML, anything needing CUDA or compiled C libraries |
| `pyenv` | Python versions only | — | Managing multiple Python installs alongside any package tool |

---

### Decision flowchart

```
Are you learning Python or writing a one-off script?
  └─ Yes → pip + venv  (built-in, no extra install needed)

Are you on a data science / ML stack (NumPy, PyTorch, CUDA, R packages)?
  └─ Yes → conda or mamba  (handles non-Python dependencies reliably)

Are you building a library others will install via `pip install`?
  └─ Yes → poetry  (best-in-class packaging + publishing workflow)

Starting a new application project (web app, CLI tool, service)?
  └─ Yes → uv  (fastest, handles deps + envs + Python versions in one tool)

Joining an existing project?
  └─ Match whatever the project already uses — check for:
       pyproject.toml with [tool.poetry] → poetry
       pyproject.toml with [tool.uv]    → uv
       requirements.in / requirements.txt → pip-tools or plain pip
       environment.yml                  → conda
```

---

### When to use each tool

#### `pip` + `venv` — the built-in baseline
Use when you're learning, prototyping, or the project is small enough that reproducibility isn't a concern yet. Every Python install already has these; there's nothing extra to set up.

**Limitation:** `pip freeze` captures your entire environment, not just direct dependencies, so requirements files become noisy and hard to maintain.

#### `pip-tools` — pip with proper lockfiles
Adds two commands on top of pip: `pip-compile` (turns `requirements.in` into a pinned `requirements.txt`) and `pip-sync` (makes the env match that file exactly). Good upgrade path from plain pip without switching ecosystems.

**Use when:** your team is already comfortable with pip but needs reproducible installs for CI.

#### `uv` — the modern default
A Rust-based drop-in replacement for pip and venv, 10–100× faster. Manages Python versions, virtual environments, and dependencies in a single tool. Generates a `uv.lock` file automatically.

```bash
uv init my-project       # new project
uv add requests          # install + update uv.lock
uv sync                  # exact reproduce from uv.lock
uv run python script.py  # run without activating the env
```

**Use when:** starting any new project. The speed difference is immediately noticeable.

#### `poetry` — for library authors
Poetry shines when you need to publish a package to PyPI. It handles version constraints, dependency groups (dev/test/docs), and the build/publish steps in one tool.

```bash
poetry new my-library
poetry add httpx
poetry add --group dev pytest
poetry publish
```

**Use when:** you are building something other people will `pip install`.

#### `conda` / `mamba` — for scientific computing
Conda manages non-Python packages (C libraries, CUDA, R) alongside Python ones, which pip cannot do reliably. `mamba` is a faster drop-in replacement for the `conda` command.

```bash
conda create -n ml-env python=3.11
conda activate ml-env
conda install pytorch torchvision -c pytorch
```

**Use when:** your project depends on native compiled libraries — especially ML frameworks with GPU support.

---

### Mixing tools: what's safe and what isn't

- `pyenv` pairs with any tool — it only manages which Python binary is active.
- `conda` and `pip` can coexist inside a conda environment, but install conda packages first and pip packages last to avoid conflicts.
- Do **not** mix `poetry` and `uv` in the same project — they both own `pyproject.toml` and will overwrite each other's metadata.
- `venv` created by `uv` is a standard venv — you can activate it and use plain `pip` inside if needed.

## Quick reference

```bash
# Create and activate a virtual environment
python -m venv .venv
source .venv/bin/activate          # macOS / Linux
.venv\Scripts\activate             # Windows

# Install packages
pip install requests httpx

# Pin dependencies
pip freeze > requirements.txt

# Restore on another machine
pip install -r requirements.txt

# --- Modern alternative with uv ---
uv init my-project
uv add requests httpx
uv sync                            # reproducible install from uv.lock
```